# Lesson 03 - Agentic Design Patterns

W tej lekcji poznajemy trzy podstawowe wzorce projektowe do budowania skutecznych agentów AI:

1. **Jasne instrukcje dla agenta** — Tworzenie precyzyjnych, definiujących rolę promptów, które kierują zachowaniem agenta
2. **Strukturalny output z modelami Pydantic** — Zapewnienie, że agenci zwracają przewidywalne, zweryfikowane dane
3. **Agenci o pojedynczej odpowiedzialności** — Projektowanie skoncentrowanych agentów, którzy dobrze wykonują jedną funkcję

Zastosujemy każdy wzorzec w scenariuszu **rekomendowania miejsc podróży**, stopniowo budując system, który potrafi sugerować destynacje, sprawdzać dostępność i obsługiwać logistykę.


## Instalacja


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Wzorzec 1: Jasne instrukcje dla agenta

Najbardziej efektywny wzorzec jest również najprostszy: pisanie jasnych, szczegółowych instrukcji dla twojego agenta.

Dobre instrukcje definiują:
- **Kim** jest agent (persona i ton)
- **Co** powinien robić (odpowiedzialności krok po kroku)
- **Jak** powinien się zachowywać (ograniczenia i styl)

Poniżej tworzymy agenta konsjerża podróży z wyraźnymi instrukcjami, które kształtują każdą jego odpowiedź.


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Wzorzec 2: Strukturalizowany wyjście za pomocą modeli Pydantic

Tekst swobodny jest przydatny do rozmowy, ale systemy docelowe potrzebują danych strukturalnych.
Łącząc **modele Pydantic** z **funkcją narzędziową**, możemy:

- Zdefiniować dokładny schemat wyjścia agenta
- Automatycznie weryfikować odpowiedzi
- Niezawodnie integrować wyniki agenta z logiką aplikacji

Przedstawiamy także narzędzie, które zwraca szczegóły miejsca docelowego, aby agent opierał swoje rekomendacje na rzeczywistych danych.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Wzorzec 3: Agenci o pojedynczej odpowiedzialności

Złożone zadania zyskują na podziale pracy między wieloma wyspecjalizowanymi agentami, z których każdy ma jedną odpowiedzialność:

- **Ekspert ds. Destynacji**, który zna się na miejscach i dostępności
- **Planista Logistyki**, który zajmuje się lotami, hotelami i planami podróży

To odzwierciedla zasadę inżynierii oprogramowania *separacji obowiązków* — każdego agenta jest łatwiej testować, utrzymywać i ulepszać niezależnie.


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Podsumowanie

W tej lekcji zastosowaliśmy trzy wzorce projektowe agentów w scenariuszu rekomendacji podróży:

| Wzorzec | Główna idea | Korzyść |
|---|---|---|
| **Jasne instrukcje** | Określ personę, obowiązki i ograniczenia na początku | Spójne, zgodne z marką zachowanie agenta |
| **Strukturalny wynik** | Użyj modeli Pydantic jako formatu odpowiedzi | Zweryfikowane, maszynowo czytelne wyniki |
| **Pojedyncza odpowiedzialność** | Nadaj każdemu agentowi jedno skoncentrowane zadanie | Łatwiejsze testowanie, utrzymanie i komponowanie |

Te wzorce naturalnie się łączą — możesz połączyć jasne instrukcje ze strukturalnym wynikiem wewnątrz agenta o pojedynczej odpowiedzialności, aby budować solidne systemy gotowe do produkcji.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Zastrzeżenie**:
Niniejszy dokument został przetłumaczony za pomocą usługi tłumaczenia AI [Co-op Translator](https://github.com/Azure/co-op-translator). Choć dążymy do dokładności, prosimy pamiętać, że automatyczne tłumaczenia mogą zawierać błędy lub niedokładności. Oryginalny dokument w jego języku źródłowym należy uznawać za autorytatywne źródło. W przypadku informacji krytycznych zalecane jest skorzystanie z profesjonalnego tłumaczenia wykonanego przez człowieka. Nie ponosimy odpowiedzialności za jakiekolwiek nieporozumienia lub błędne interpretacje wynikające z użycia tego tłumaczenia.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
